# HAI Repeated Experiments (Colab Runner)

This notebook is designed to run the heavy pipeline in a Google Colab runtime from VS Code.

It executes:
1. run_multicollinearity_cleanup.py
2. run_repeated_experiments.py (20 seeds, checkpoint/resume enabled)

Setup behavior in Cell 2:
- Mounts Google Drive (Colab only)
- Auto-clones the public GitHub repository into /content/HAI_April_2026
- Falls back to manual path selection if needed

Repository URL: https://github.com/ZahirAhmadChaudhry/HAI_April_2026

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/ZahirAhmadChaudhry/HAI_April_2026.git'
REPO_NAME = 'HAI_April_2026'

# Optional manual override:
# MANUAL_PROJECT_DIR = Path('/content/drive/MyDrive/HAI_April_2026')
MANUAL_PROJECT_DIR = None

clone_target = Path('/content') / REPO_NAME
if IN_COLAB and not clone_target.exists():
    print('Cloning public repository into', clone_target)
    try:
        subprocess.check_call(['git', 'clone', REPO_URL, str(clone_target)])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Git clone failed. Check network or set MANUAL_PROJECT_DIR to an existing local copy.'
        ) from exc

candidates = [
    MANUAL_PROJECT_DIR,
    clone_target,
    Path('/content/drive/MyDrive/HAI_April_2026'),
    Path('/content/HAI_April_2026'),
    Path.cwd(),
]

project_dir = None
for c in candidates:
    if c is None:
        continue
    c = Path(c)
    if (c / 'clean_hai_dataset.csv').exists() and (c / 'run_repeated_experiments.py').exists():
        project_dir = c
        break

if project_dir is None:
    raise FileNotFoundError(
        'Could not find HAI_April_2026. Set MANUAL_PROJECT_DIR to the correct path.'
    )

os.chdir(project_dir)
print('Project directory:', project_dir)
print('Python executable:', sys.executable)
print('Current working directory:', Path.cwd())

Mounted at /content/drive


FileNotFoundError: Could not find HAI_April_2026. Set MANUAL_PROJECT_DIR to the correct folder path.

In [ ]:
import subprocess
import sys

packages = [
    'numpy',
    'pandas',
    'scikit-learn',
    'imbalanced-learn',
    'optuna',
    'xgboost',
    'lightgbm',
    'catboost',
    'shap',
    'joblib',
    'matplotlib',
    'seaborn',
    'scipy'
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed.')

In [ ]:
import subprocess
import sys

print('Running multicollinearity cleanup...')
subprocess.check_call([sys.executable, 'run_multicollinearity_cleanup.py'], cwd=project_dir)
print('Part 1 complete.')

In [ ]:
import json
import pandas as pd
from pathlib import Path

cleaned_path = Path(project_dir) / 'cleaned_feature_sets.json'
report_path = Path(project_dir) / 'multicollinearity_cleanup_report.md'
heatmap_path = Path(project_dir) / 'figures' / 'correlation_matrix.png'

payload = json.loads(cleaned_path.read_text(encoding='utf-8'))
print('Model A cleaned feature count:', len(payload['model_a_cleaned']))
print('Model B cleaned feature count:', len(payload['model_b_cleaned']))
print('Dropped features:', list(payload['dropped_features'].keys()))
print('Heatmap exists:', heatmap_path.exists())

print('\nReport preview:')
print('-' * 80)
print(report_path.read_text(encoding='utf-8')[:2000])

In [ ]:
import subprocess
import sys

# Set to True to run the full 20-seed repeated experiment (can take hours).
RUN_FULL_REPEATED_EXPERIMENT = True

if RUN_FULL_REPEATED_EXPERIMENT:
    print('Running repeated experiments (20 seeds, checkpoint-enabled)...')
    subprocess.check_call([sys.executable, 'run_repeated_experiments.py'], cwd=project_dir)
    print('Part 2 complete.')
else:
    print('Skipped Part 2. Set RUN_FULL_REPEATED_EXPERIMENT=True when ready.')

In [ ]:
from pathlib import Path
import pandas as pd

expected_files = [
    'repeated_experiment_results.csv',
    'repeated_experiment_summary.md',
    'repeated_shap_stability.csv',
    'figures/auc_distribution_boxplot.png',
    'figures/auc_difference_histogram.png',
    'figures/shap_stability_plot.png'
]

missing = []
for rel in expected_files:
    p = Path(project_dir) / rel
    if not p.exists():
        missing.append(rel)

if missing:
    print('Missing outputs:')
    for m in missing:
        print('-', m)
else:
    print('All repeated-experiment outputs are present.')

results_path = Path(project_dir) / 'repeated_experiment_results.csv'
if results_path.exists():
    df = pd.read_csv(results_path)
    print('Results shape:', df.shape)
    print('Unique seeds:', sorted(df['seed'].unique().tolist())[:5], '...')
    print('Model rows per seed (should be 8):')
    print(df.groupby('seed').size().describe())
    display(df.head())

## Notes

- run_repeated_experiments.py uses checkpointing via repeated_experiment_checkpoint.csv.
- If runtime disconnects, re-run Cell 6; completed seeds are skipped automatically.
- The repository is public, so token setup is not required for cloning.
- After completion, check repeated_experiment_summary.md for paper-ready aggregate results.